<h2 align='center'>ML Flow Tutorial with four different models: MUTLI-EXPERIMENT MODEL TRACKING</h2>

Link: https://youtu.be/6ngxBkx05Fs?si=c7vIJ9X1MsYURoy_

This notebook includes -

1. Model Tracking
2. Model Registry
3. Model Loading
4. Moving everything to Production using MLFlow Client (Previously all the models were logged in the MLFlow UI. Now we will move the best model to production using MLFlow Client)

In [31]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# MLFlow Libraries
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models import infer_signature

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [25]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print(X_train)
print(y_train)

[[ 1.18673836  1.51144074  0.78490373 ... -0.61229492 -0.13830257
  -0.24753395]
 [-1.28810271 -1.03855344 -2.07092052 ...  0.49607021 -1.50376955
   0.62474155]
 [ 1.66393774  1.55142135  2.25024183 ... -0.69955586  1.36600648
  -0.68290518]
 ...
 [ 0.43101615  0.90013637 -0.42606094 ... -0.32069604 -1.01508454
   0.11782042]
 [ 0.79839935  1.5003473  -0.45098948 ... -0.54728572 -1.42140103
   0.11944858]
 [ 0.67367695  1.27538516 -0.39960348 ... -0.464427   -1.22522394
   0.10635794]]
[0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1
 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0
 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

### Experiment 1: Train Logistic Regression Classifier

In [4]:
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



### Experiment 2: Train Random Forest Classifier

In [5]:
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       270
           1       0.95      0.63      0.76        30

    accuracy                           0.96       300
   macro avg       0.96      0.81      0.87       300
weighted avg       0.96      0.96      0.96       300



### Experiment 3: Train XGBoost

In [6]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



### Experiment 4: Handle class imbalance using SMOTETomek and then Train XGBoost

In [7]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [8]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



<h2 align="center" style="color:blue">Track Experiments Using MLFlow</h2>

In [10]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [11]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [12]:
reports

[{'0': {'precision': 0.9454545454545454,
   'recall': 0.9629629629629629,
   'f1-score': 0.9541284403669725,
   'support': 270.0},
  '1': {'precision': 0.6,
   'recall': 0.5,
   'f1-score': 0.5454545454545454,
   'support': 30.0},
  'accuracy': 0.9166666666666666,
  'macro avg': {'precision': 0.7727272727272727,
   'recall': 0.7314814814814814,
   'f1-score': 0.749791492910759,
   'support': 300.0},
  'weighted avg': {'precision': 0.9109090909090909,
   'recall': 0.9166666666666666,
   'f1-score': 0.91326105087573,
   'support': 300.0}},
 {'0': {'precision': 0.9711191335740073,
   'recall': 0.9962962962962963,
   'f1-score': 0.9835466179159049,
   'support': 270.0},
  '1': {'precision': 0.9565217391304348,
   'recall': 0.7333333333333333,
   'f1-score': 0.8301886792452831,
   'support': 30.0},
  'accuracy': 0.97,
  'macro avg': {'precision': 0.963820436352221,
   'recall': 0.8648148148148148,
   'f1-score': 0.906867648580594,
   'support': 300.0},
  'weighted avg': {'precision': 0.9696

### **Tracking Multiple Experiments with MLFlow**

In [13]:
# Initialize a MLFlow instance that will run in MLFlow cloud
# !mlflow server --host 127.0.0.1 --port 5000

^C


In [ ]:
# Starting MLFlow UI
!mlflow ui

In [ ]:
# Initialize MLflow
mlflow.set_experiment("Binary Classification")
mlflow.set_tracking_uri("http://127.0.0.1:5000/")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name = model_name):
        mlflow.log_param("model", model_name)     
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        signature = infer_signature(X_test, y_test)
        
        if "XGB" in model_name:
            model_info = mlflow.xgboost.log_model(model, model_name = model_name, signature = signature, input_example = X_test)
        else:
            model_info = mlflow.sklearn.log_model(sk_model = model, model_name = model_name, signature = signature, input_example = X_test) 

Registered model 'Logistic Regression' already exists. Creating a new version of this model...
2026/03/11 16:47:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Logistic Regression, version 2
Created version '2' of model 'Logistic Regression'.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/3/runs/4d2f54bd6e9144849748f64962314a55
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Registered model 'Random Forest' already exists. Creating a new version of this model...
2026/03/11 16:48:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Random Forest, version 2
Created version '2' of model 'Random Forest'.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/3/runs/763e721cbb5a4fe59ee1700b101f82ac
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/11 16:48:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registered model 'XGBClassifier' already exists. Creating a new version of this model...
2026/03/11 16:48:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBClassifier, version 2
Created version '2' of model 'XGBClassifier'.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/3/runs/d4357621ba80438eab287e3c027d7b1a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/11 16:48:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registered model 'XGBClassifier With SMOTE' already exists. Creating a new version of this model...
2026/03/11 16:48:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBClassifier With SMOTE, version 2
Created version '2' of model 'XGBClassifier With SMOTE'.


🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/3/runs/99271aab37974a119bb61965f20ce662
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


### **Register the Model**
We only register the best model

In [ ]:
# Here we will register the model and provide the run-id manually
model_name = 'XGB-Smote'
run_id=input('Please type RunID')
model_uri = f'runs:/{run_id}/model'

# To get latest Model-URI
# model_info.model_uri

# with mlflow.start_run(run_id = run_id):
#     mlflow.register_model(model_uri = model_uri, name = model_name)

result = mlflow.register_model(model_uri, model_name)

### **Load the Model (Manually providing the Run-id)**
We are loading the model that performs the best

In [ ]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
print("Loaded model info:",loaded_model)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

### **Loading the Model as a function**

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
print("Loaded model info:",loaded_model)

# Predicting on X_test, then adding y_test and predictions made on X_test to a dataframe to compare the result
predictions = loaded_model.predict(X_test)
print("Predictions:", predictions)
pred_df = pd.DataFrame(y_test, columns = ['y_test'])
pred_df['pred_on_y_test'] = predictions
#pred_df.head()

# Comparing y_test and Predictions on X_test -> if ytest = preds then 1 else 0
pred_df['ytest_pred_com'] = np.where(pred_df['y_test']==pred_df['pred_on_y_test'], 1,0)
pred_df['ytest_pred_com'].value_counts()

Loaded model info: mlflow.pyfunc.loaded_model:
  artifact_path: file:d:/OneDrive - Northeastern University/Jupyter Notebook/Machine
    Learning Algorithms/5 - Model Deployment and MLOps/03 - MLFlow/mlruns/3/models/m-29711397874347df8405f0ca6b3f7aac/artifacts
  flavor: mlflow.xgboost
  run_id: 99271aab37974a119bb61965f20ce662

Predictions: [0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0
 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 1
 1 0 0 0]


### Moving everything to Production using MLFlow Client
Previously all the models were logged in the MLFlow UI. Now we will move the best model to production using MLFlow Client.

In [24]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"
production_model_name = "binary-classification-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri = model_uri, dst_name = production_model_name)

Successfully registered model 'binary-classification-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'binary-classification-prod'.


<ModelVersion: aliases=[], creation_timestamp=1770342913756, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1770342913756, metrics=None, model_id=None, name='binary-classification-prod', params=None, run_id='7523c184280f45779f7ea650c7ca4c67', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1'>